# Triage Dataset Modifications
This notebook looks at modifying `patient_priority.csv`, which is a dataset of patients with different background on a 4-level systems (Red, Orange, Yellow, Green). The Triage system can be found at `https://link.springer.com/article/10.1186/1757-7241-20-28`, with the following definitions.
- **Red:** Resuscitation -> Immediate life-saving intervention needed (e.g., cardiac arrest).
- **Orange:** Emergency -> Very urgent, high-risk, or time-sensitive condition (e.g., chest pain, stroke).
- **Yellow:** Urgent -> Stable, but requires urgent assessment and treatment (e.g., abdominal pain).
- **Green:** Semi-Urgent -> Less urgent, lower risk (e.g., sprained ankle).

Modifications are performed to change the dataset to fit a `General Practitioner (G.P.)` settings (i.e., Removing data columns not commonly available in those settings, Reduce complexity levels in patient categorisation). These are the main steps taken in the modifications.
1. **Setting up environment:** Setting up the environment
2. **Load Data:** Load the original dataset for modifications
3. **Dropping data:** Drop rows with missing target variables
4. **Recategorising based on urgency:** Understanding the data and cleaning missing data
5. **Save Modified Dataset:** Saving modified dataset (Optional)

**Note:** This notebook is not intended to perform clean any missing data in `X_variables`.

## Step 1: Setting up environment
This steps sets up the environment for the rest of the sections through the following sub-steps.
- Resolves imports
- Set environment variables and configurations (e.g., data_filename)
- Safely ensures that the paths used in the notebook are all created.

In [55]:
# File_name to extract original raw data and save file
data_filename: str = "patient_priority.csv"
save_filename: str = "patient_priority_modified.csv"

# Set option on whether to save modified file. Defaults to True
save_file: bool = True

In [56]:
import os

CWD = os.getcwd()
ROOT = os.path.abspath(os.path.join(CWD, ".."))


# Create Function to create a directory
def create_folder(relative_path: str):
    """
    Helper function to create a folder if it does not exists and return file path

    Args:
        relative_path (str): Relative Path with respect to root

    Returns:
        str: String containing absolute file path
    """
    path = os.path.join(ROOT, relative_path)
    try:
        os.makedirs(path, exist_ok=True)
        print(f"Ensured directory '{path}' exists")
    except OSError as exc:
        print(f"Exception faced when creating folder '{path}': {exc}")
    return os.path.abspath(path)


DATA_DIR = create_folder(relative_path="data")

Ensured directory 'c:\DBTT\DBTT_G5T1\data' exists


## Step 2: Load Data

In [57]:
import os

import pandas as pd
from pandas import DataFrame


def extract_dataframe_from_file(file_name: str, dir_path: str) -> dict[str, DataFrame]:
    """
    Finds the Data directory for file with specified filename and retrieve as a DataFrame

    Args:
        filename (str): Name that the filename should contain.
        dir_path (str): Path of directory containing filename

    Raises:
        ValueError: If filename is invalid, or filetype is wrong
        FileNotFoundError: if file is not found in the directory

    Returns:
        dict[str, DataFrame]: Dictionary mapping sheet_name to DataFrame. For CSV, it maps the filename to the DataFrame.
    """
    if not file_name.lower().endswith((".csv", ".xlsx", ".xls")):
        raise ValueError(f"Invalid filename received: {file_name}")

    # Build full absolute path
    file_path = os.path.abspath(os.path.join(dir_path, file_name))

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found in '{dir_path}': {file_name}")

    # Instantiate dictionary to manage mapping
    dfs: dict[str, DataFrame] = {}

    # Load file based on extension
    if file_name.lower().endswith(".csv"):
        dfs[file_name] = pd.read_csv(file_path, index_col=False)

    else:
        # Open the Excel file
        xls = pd.ExcelFile(file_path)

        # Loop through each sheet
        for sheet_name in xls.sheet_names:
            dfs[sheet_name] = pd.read_excel(xls, sheet_name=sheet_name, index_col=False)

    # Return dictionary
    return dfs


# Extract DataFrames and print out Dataframes names
dfs: dict[str, DataFrame] = extract_dataframe_from_file(
    file_name=data_filename, dir_path=DATA_DIR
)
print(f"List of DataFrame names: {list(dfs.keys())}")

List of DataFrame names: ['patient_priority.csv']


In [58]:
# Extract DataFrame for CSV
df = dfs.get(data_filename)
print(f"Shape of original DataFrame: {df.shape}")
df.head(5)

Shape of original DataFrame: (6962, 18)


,Unnamed: 0,age,gender,chest pain type,blood pressure,cholesterol,max heart rate,exercise angina,plasma glucose,skin_thickness,insulin,bmi,diabetes_pedigree,hypertension,heart_disease,Residence_type,smoking_status,triage
0,0,40.0,1.0,2.0,140.0,294.0,172.0,0.0,108.0,43.0,92.0,19.0,0.467386,0.0,0.0,Urban,never smoked,yellow
1,1,49.0,0.0,3.0,160.0,180.0,156.0,0.0,75.0,47.0,90.0,18.0,0.467386,0.0,0.0,Urban,never smoked,orange
2,2,37.0,1.0,2.0,130.0,294.0,156.0,0.0,98.0,53.0,102.0,23.0,0.467386,0.0,0.0,Urban,never smoked,yellow
3,3,48.0,0.0,4.0,138.0,214.0,156.0,1.0,72.0,51.0,118.0,18.0,0.467386,0.0,0.0,Urban,never smoked,orange
4,4,54.0,1.0,3.0,150.0,195.0,156.0,0.0,108.0,90.0,83.0,21.0,0.467386,0.0,0.0,Urban,never smoked,yellow


## Step 3: Dropping data

**Dropping irrelevant columns**: This subsection drops the columns that are either not meaningful for classification (i.e., index col) or not available in G.P. context (i.e., cholesterol, plasma glucose, skin_thickness, insulin)

In [59]:
# Drop Index column
df_cols: int = df.shape[1]
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

# Drop unnecessary columns
drop_columns = [
    "gender",
    "cholesterol",
    "plasma glucose",
    "skin_thickness",
    "insulin",
    "Residence_type",
    "diabetes_pedigree",
]
df = df.drop(columns=drop_columns, errors="ignore")

# View Updated DataFrame
print(f"Removed {df_cols - df.shape[1]} columns")
print(f"Shape of new DataFrame: {df.shape}")
df.head(5)

Removed 8 columns
Shape of new DataFrame: (6962, 10)


,age,chest pain type,blood pressure,max heart rate,exercise angina,bmi,hypertension,heart_disease,smoking_status,triage
0,40.0,2.0,140.0,172.0,0.0,19.0,0.0,0.0,never smoked,yellow
1,49.0,3.0,160.0,156.0,0.0,18.0,0.0,0.0,never smoked,orange
2,37.0,2.0,130.0,156.0,0.0,23.0,0.0,0.0,never smoked,yellow
3,48.0,4.0,138.0,156.0,1.0,18.0,0.0,0.0,never smoked,orange
4,54.0,3.0,150.0,156.0,0.0,21.0,0.0,0.0,never smoked,yellow


**Dropping target**: This subsection drops missing 'triage' values to prepare for the target conversion in the next step.

In [60]:
# Drop rows that contain missing 'triage'
df_rows: int = df.shape[0]
df = df.dropna(subset=["triage"])

# View updated dataframe
print(f"Removed rows with missing 'triage' values: {df_rows - df.shape[0]}")
print(f"Shape of new DataFrame: {df.shape}")
df.head(5)

Removed rows with missing 'triage' values: 410
Shape of new DataFrame: (6552, 10)


,age,chest pain type,blood pressure,max heart rate,exercise angina,bmi,hypertension,heart_disease,smoking_status,triage
0,40.0,2.0,140.0,172.0,0.0,19.0,0.0,0.0,never smoked,yellow
1,49.0,3.0,160.0,156.0,0.0,18.0,0.0,0.0,never smoked,orange
2,37.0,2.0,130.0,156.0,0.0,23.0,0.0,0.0,never smoked,yellow
3,48.0,4.0,138.0,156.0,1.0,18.0,0.0,0.0,never smoked,orange
4,54.0,3.0,150.0,156.0,0.0,21.0,0.0,0.0,never smoked,yellow


## Step 4: Recategorising based on urgency
Convert the 'triage' values to a binary encoded 'y' variable on whether urgent care is required.
- **0:** Represents non-urgent patients that maps to 'green' and 'yellow' category.
- **1:** Represents urgent patient that maps to 'red' and 'orange' categories.

In [61]:
# Create new label 'y' to represent higher urgency of the patient
df["y"] = df["triage"].isin(["red", "orange"]).astype(int)

# Drop 'triage' col and view df
df = df.drop(columns=["triage"], errors="ignore")
print(f"Shape of new DataFrame: {df.shape}")
df.head(5)

Shape of new DataFrame: (6552, 10)


,age,chest pain type,blood pressure,max heart rate,exercise angina,bmi,hypertension,heart_disease,smoking_status,y
0,40.0,2.0,140.0,172.0,0.0,19.0,0.0,0.0,never smoked,0
1,49.0,3.0,160.0,156.0,0.0,18.0,0.0,0.0,never smoked,1
2,37.0,2.0,130.0,156.0,0.0,23.0,0.0,0.0,never smoked,0
3,48.0,4.0,138.0,156.0,1.0,18.0,0.0,0.0,never smoked,1
4,54.0,3.0,150.0,156.0,0.0,21.0,0.0,0.0,never smoked,0


In [62]:
# Understanding values distribution of target
df["y"].value_counts()

y
0    6077
1     475
Name: count, dtype: int64

## Step 5: Save Modified Dataset

In [63]:
if save_file and save_filename:
    save_file_path = os.path.abspath(os.path.join(DATA_DIR, save_filename))
    print(f"Saving DataFrame to filename: {save_filename}")
    df.to_csv(save_file_path, index=False)
    print("Successfully saved DataFrame to csv")

Saving DataFrame to filename: patient_priority_modified.csv


Successfully saved DataFrame to csv
